In [1]:
from cloudvolume import CloudVolume
import numpy as np
import pathlib
import networkx as nx
import json

/Users/mateljk1/Documents/projects/aplbrain/zebrafish-pencilsdown/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SVOX_CLOUDPATH = "precomputed://s3://cvdb-bossdb-boss/smith2024/zebrafish/supervoxels"
COMPONENT_PATH = pathlib.Path("./component-graphs/")


In [3]:
svox = CloudVolume(SVOX_CLOUDPATH, mip=0)

In [4]:
size_in_bytes = np.prod(svox.shape) * np.dtype(svox.dtype).itemsize
size_in_gb = size_in_bytes / (1024**3)
size_in_gb

np.float64(6808.823103904724)

In [5]:
for gpath in sorted(COMPONENT_PATH.glob("*.gexf")):
    editcount = int(str(gpath).split(".")[-2])
    remap_dict = {}
    g = nx.read_gexf(gpath)
    # components:
    cc = nx.connected_components(g)
    # cc = [[int(i) for i in c] for c in cc]
    for comp in cc:
        min_seg = min(comp)
        for seg in comp:
            remap_dict[seg] = min_seg

    with open(str(gpath) + ".remap.json", 'w') as fh:
        sorted_remap_dict = {k: v for k, v in sorted(remap_dict.items())}
        json.dump(sorted_remap_dict, fh, indent=2)